In [1]:
import pandas as pd
import os
import time
import re
from dotenv import load_dotenv
from openai import OpenAI

#.env에서 OpenAI API 키 불러오기
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(".env에 OPENAI_API_KEY가 설정되지 않았습니다.")

client = OpenAI(api_key=openai_api_key)

In [1]:
import pandas as pd
import os
import time
import kss
from openai import OpenAI

# OpenAI API 설정
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 문장 단위 역번역 함수
def gpt_back_translate_en(text):
    try:
        print(f"역번역 시작: {text[:30]}...")
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례와 일반 통화 사례가 섞여있어. "
                        "너의 임무는 이 문장을 먼저 영어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "**최종 출력은 번역된 한국어 문장만 보여줘.** 최종 결과에 영어가 나오면 실패. 영어 텍스트 삭제"
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 영어로 번역한 후 다시 한국어로 자연스럽게 재번역해줘. 최종출력은 영어 문장만:\n\n'{text}'",
                },
            ],
            temperature=0.9,
            max_tokens=500,
        )
        result = response.choices[0].message.content.strip()
        print(f"역번역 완료: {result[:30]}...")
        return result
    except Exception as e:
        print(f"[오류] 역번역 실패: {e}")
        return None

# 전체 문장을 문장 단위로 나눠서 번역 후 합치는 함수
def back_translate_paragraph(text):
    sentences = kss.split_sentences(text)
    translated_sentences = []
    for sent in sentences:
        translated = gpt_back_translate_en(sent)
        if translated is None:
            translated = ""
        translated_sentences.append(translated.strip())
        time.sleep(1.0)  # 과부하 방지
    return " ".join(translated_sentences)

# 경로 설정
INPUT_FILE = "C:/Users/user/Downloads/보이스피싱/woogawooga_project/dataset_create/Chaeyeon/시나리오통화테스트셋수정0724.csv"
OUTPUT_FILE = "시나리오수정영어증강.csv"
FAILED_FILE = "failed_log.csv"

# 처리된 file_id 불러오기
def load_processed_ids():
    if not os.path.exists(OUTPUT_FILE):
        print("처리된 파일 없음 (새 시작)")
        return set()
    try:
        df_done = pd.read_csv(OUTPUT_FILE)
        if 'file_id' not in df_done.columns:
            print("'file_id' 컬럼이 없음")
            return set()
        processed = set(df_done['file_id'].dropna().unique())
        print(f"처리된 file_id 수: {len(processed)}개")
        return processed
    except Exception as e:
        print(f"처리된 목록 로딩 실패: {e}")
        return set()

# CSV 헤더 생성
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'w', encoding='utf-8', newline='') as f:
        f.write("file_id,is_phishing,text,back_translated\n")

if not os.path.exists(FAILED_FILE):
    with open(FAILED_FILE, 'w', encoding='utf-8', newline='') as f:
        f.write("file_id,is_phishing,text,error_type\n")

# 전체 데이터 로딩
df = pd.read_csv(INPUT_FILE)
grouped = df.groupby("file_id")

processed_ids = load_processed_ids()

# 메인 처리 루프
for file_id, group in grouped:
    if file_id in processed_ids:
        print(f"건너뜀 (이미 처리됨): {file_id}")
        continue

    is_phishing = group['is_phishing'].iloc[0]

    for _, row in group.iterrows():
        text = str(row['text']).strip()
        translated = back_translate_paragraph(text)

        if translated is None:
            with open(FAILED_FILE, 'a', encoding='utf-8') as f_fail:
                f_fail.write(f"{file_id},{is_phishing},{text},gpt_fail\n")
            translated = ""

        # CSV 안전 저장용 정리
        text_clean = text.replace("\n", " ").replace(",", " ")
        translated_clean = translated.replace("\n", " ").replace(",", " ")

        with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
            f_out.write(f"{file_id},{is_phishing},{text_clean},{translated_clean}\n")

        time.sleep(0.5)

    print(f"처리 완료: {file_id}")
    processed_ids.add(file_id)



[Kss]: Because there's no supported C++ morpheme analyzer, Kss will take pecab as a backend. :D
For your information, Kss also supports mecab backend.
We recommend you to install mecab or konlpy.tag.Mecab for faster execution of Kss.
Please refer to following web sites for details:
- mecab: https://cleancode-ws.tistory.com/97
- konlpy.tag.Mecab: https://uwgdqo.tistory.com/363



처리된 file_id 수: 0개


c:\Users\user\Downloads\보이스피싱\woogawooga_project\.venv\lib\site-packages\pecab\_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]


역번역 시작: A: 여보세요?...
역번역 완료: 네, 안녕하세요?...
역번역 시작: B: 어, 엄마....
역번역 완료: 어, 엄마....
역번역 시작: 뭐해?...
역번역 완료: 뭐하고 있어?...
역번역 시작: A: 그냥 점심 준비하고 있었지....
역번역 완료: A: 그냥 점심 준비하고 있었어....
역번역 시작: 오늘 일찍 전화했네?...
역번역 완료: 오늘 일찍 연락했네?...
역번역 시작: B: 아, 회사에서 정기예금 만기 안내 문자가 와서 혹...
역번역 완료: 아, 회사에서 정기예금 만기 알림 문자가 왔는데, 엄마...
역번역 시작: A: 아, 그거 지난주에 은행에서 전화 왔더라....
역번역 완료: 아, 저거 지난주에 은행에서 전화가 왔었어요....
역번역 시작: 자동 연장되게 놔뒀어....
역번역 완료: 자동으로 연장되도록 그냥 뒀어요....
역번역 시작: 너 혹시 집에 세금 관련 서류 온 거 봤니?...
역번역 완료: 집에 세금 관련 서류가 도착했는지 혹시 확인해 봤어?...
역번역 시작: B: 어, 오늘 아침에 우편함에 있길래 책상 위에 올려...
역번역 완료: B: 아, 오늘 아침에 우편함에서 발견해서 책상 위에 ...
역번역 시작: 근데 기초공제 신청서류라고 해서 뭐가 뭔지 잘 모르겠더...
역번역 완료: 근데 기초공제 신청서류가 뭔지 잘 이해가 안 됐어요....
역번역 시작: A: 그거 작년에 네 아빠가 일했을 때 썼던 거랑 비슷...
역번역 완료: A: 그건 작년에 네 아버지가 일하실 때 사용하셨던 거...
역번역 시작: 필요하면 내가 퇴근하고 다시 한번 같이 확인해줄게....
역번역 완료: 필요하면 내가 퇴근하고 나서 다시 한 번 함께 확인해줄...
역번역 시작: B: 응, 고마워....
역번역 완료: 응, 고마워....
역번역 시작: 엄마 혹시 오늘 저녁에 뭐 먹을 건데?...
역번역 완료: 엄마, 오늘 저녁에 뭐 먹을 예정이야?...
역번역 시작: 나 늦을 것 같아서....
역번역 완료: 좀 늦을 것 같아....
역번역 시

c:\Users\user\Downloads\보이스피싱\woogawooga_project\.venv\lib\site-packages\pecab\_tokenizer.py:274: RuntimeWarning: overflow encountered in scalar add
  least_cost += word_cost


역번역 시작: A: 안녕하세요, SK텔레콤 고객센터입니다....
역번역 완료: 안녕하세요, SK텔레콤 고객센터입니다....
역번역 시작: 박지수 고객님 맞으신가요?...
역번역 완료: 박지수 고객님이신가요?...
역번역 시작: B: 네, 맞아요....
역번역 완료: 네, 그렇습니다....
역번역 시작: A: 고객님께서 사용하시는 인터넷과 TV 약정이 이번 ...
역번역 완료: 고객님, 사용하고 계신 인터넷과 TV 서비스 약정이 이...
역번역 시작: 혹시 평소 서비스 이용 중에 불편했던 점은 없으셨나요?...
역번역 완료: 서비스를 이용하시면서 불편했던 점이 있으셨나요?...
역번역 시작: B: 크게 불편하진 않았는데, 가끔 인터넷 속도가 조금...
역번역 완료: B: 크게 불편하지는 않았지만, 가끔 인터넷 속도가 좀...
역번역 시작: A: 네, 불편을 드려 죄송합니다....
역번역 완료: 네, 불편을 끼쳐드려 정말 죄송합니다....
역번역 시작: 앞으로 속도 관련 불편함이 최소화될 수 있도록 점검 요...
역번역 완료: 앞으로 속도와 관련된 불편함이 최소화되도록 점검 요청을...
역번역 시작: 혹시 현재 요금제 유지 계획이신가요, 아니면 업그레이드...
역번역 완료: 현재 사용 중인 요금제를 계속 유지하실 예정이신가요, ...
역번역 시작: B: 아직은 지금 요금제 그대로 쓰고 싶어요....
역번역 완료: B: 네, 지금 있는 요금제 계속 사용하고 싶습니다....
역번역 시작: 약정 만료되면 어떤 변화가 있는지도 궁금해서요....
역번역 완료: 약정이 끝나면 어떤 점이 달라지는지 궁금해서요....
역번역 시작: A: 약정이 끝나면 기본 요금이 소폭 인상될 수 있기 ...
역번역 완료: A: 계약 기간이 종료되면 기본 요금이 조금 오를 수 ...
역번역 시작: 특히 재약정 시 월 5,500원 추가 할인과 데이터 쿠...
역번역 완료: 특히 재약정하실 때 월 5,500원의 추가 할인과 데이...
역번역 시작: B: 상품권은 어느 정도 받을 수 있어요?...
역

In [2]:
import pandas as pd

# 원본 CSV 로딩
df = pd.read_csv("시나리오수정영어증강.csv")

# 역번역 행 생성
df_bt = df.copy()
df_bt["file_id"] = df_bt["file_id"].astype(str) + "_en"
df_bt["text"] = df_bt["back_translated"]  # 역번역 텍스트를 text로 이동

# 불필요한 컬럼 삭제 (원한다면)
df_bt = df_bt[["file_id", "is_phishing", "text"]]  # back_translated 제거
df_orig = df[["file_id", "is_phishing", "text"]]   # 원본도 동일한 형태로 정리


# 두 데이터프레임 합치기
df_combined = pd.concat([df_orig, df_bt], ignore_index=True)

# 결과 확인 (또는 저장)
df_combined.to_csv("시나리오수정증강합본1.csv", index=False, encoding="utf-8-sig")


# 중국어 증강

In [3]:
import pandas as pd
import os
import time
import kss
from openai import OpenAI

# OpenAI API 설정
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 문장 단위 역번역 함수
def gpt_back_translate_en(text):
    try:
        print(f"역번역 시작: {text[:30]}...")
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례와 일반 통화 사례가 섞여있어. "
                        "너의 임무는 이 문장을 먼저 중국어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "**최종 출력은 번역된 한국어 문장만 보여줘.** 최종 결과에 중국어가 나오면 실패. 중국어 텍스트 삭제"
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 중국어로 번역한 후 다시 한국어로 자연스럽게 재번역해줘. 최종출력은 한국어 문장만:\n\n'{text}'",
                },
            ],
            temperature=0.9,
            max_tokens=500,
        )
        result = response.choices[0].message.content.strip()
        print(f"역번역 완료: {result[:30]}...")
        return result
    except Exception as e:
        print(f"[오류] 역번역 실패: {e}")
        return None

# 전체 문장을 문장 단위로 나눠서 번역 후 합치는 함수
def back_translate_paragraph(text):
    sentences = kss.split_sentences(text)
    translated_sentences = []
    for sent in sentences:
        translated = gpt_back_translate_en(sent)
        if translated is None:
            translated = ""
        translated_sentences.append(translated.strip())
        time.sleep(1.0)  # 과부하 방지
    return " ".join(translated_sentences)

# 경로 설정
INPUT_FILE = "C:/Users/user/Downloads/보이스피싱/woogawooga_project/dataset_create/Chaeyeon/시나리오통화테스트셋수정0724.csv"
OUTPUT_FILE = "시나리오수정중국어증강.csv"
FAILED_FILE = "failed_log.csv"

# 처리된 file_id 불러오기
def load_processed_ids():
    if not os.path.exists(OUTPUT_FILE):
        print("처리된 파일 없음 (새 시작)")
        return set()
    try:
        df_done = pd.read_csv(OUTPUT_FILE)
        if 'file_id' not in df_done.columns:
            print("'file_id' 컬럼이 없음")
            return set()
        processed = set(df_done['file_id'].dropna().unique())
        print(f"처리된 file_id 수: {len(processed)}개")
        return processed
    except Exception as e:
        print(f"처리된 목록 로딩 실패: {e}")
        return set()

# CSV 헤더 생성
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'w', encoding='utf-8', newline='') as f:
        f.write("file_id,is_phishing,text,back_translated\n")

if not os.path.exists(FAILED_FILE):
    with open(FAILED_FILE, 'w', encoding='utf-8', newline='') as f:
        f.write("file_id,is_phishing,text,error_type\n")

# 전체 데이터 로딩
df = pd.read_csv(INPUT_FILE)
grouped = df.groupby("file_id")

processed_ids = load_processed_ids()

# 메인 처리 루프
for file_id, group in grouped:
    if file_id in processed_ids:
        print(f"건너뜀 (이미 처리됨): {file_id}")
        continue

    is_phishing = group['is_phishing'].iloc[0]

    for _, row in group.iterrows():
        text = str(row['text']).strip()
        translated = back_translate_paragraph(text)

        if translated is None:
            with open(FAILED_FILE, 'a', encoding='utf-8') as f_fail:
                f_fail.write(f"{file_id},{is_phishing},{text},gpt_fail\n")
            translated = ""

        # CSV 안전 저장용 정리
        text_clean = text.replace("\n", " ").replace(",", " ")
        translated_clean = translated.replace("\n", " ").replace(",", " ")

        with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
            f_out.write(f"{file_id},{is_phishing},{text_clean},{translated_clean}\n")

        time.sleep(0.5)

    print(f"처리 완료: {file_id}")
    processed_ids.add(file_id)


처리된 file_id 수: 0개
역번역 시작: A: 여보세요?...
역번역 완료: A: 네, 안녕하세요?...
역번역 시작: B: 어, 엄마....
역번역 완료: B: 어, 엄마....
역번역 시작: 뭐해?...
역번역 완료: 어떤 일 하고 있어?...
역번역 시작: A: 그냥 점심 준비하고 있었지....
역번역 완료: A: 그냥 점심을 준비하고 있었어....
역번역 시작: 오늘 일찍 전화했네?...
역번역 완료: 오늘 왜 이렇게 일찍 전화했어?...
역번역 시작: B: 아, 회사에서 정기예금 만기 안내 문자가 와서 혹...
역번역 완료: B: 아, 회사에서 정기예금 만료 안내 문자 왔는데, ...
역번역 시작: A: 아, 그거 지난주에 은행에서 전화 왔더라....
역번역 완료: A: 아, 그거 지난주에 은행에서 전화가 왔었어....
역번역 시작: 자동 연장되게 놔뒀어....
역번역 완료: 자동으로 갱신되도록 그냥 뒀어....
역번역 시작: 너 혹시 집에 세금 관련 서류 온 거 봤니?...
역번역 완료: "집에 세금 관련 서류가 도착했는지 확인해 봤어?"...
역번역 시작: B: 어, 오늘 아침에 우편함에 있길래 책상 위에 올려...
역번역 완료: B: 오늘 아침에 우편함에 있길래 책상 위에 올려뒀어....
역번역 시작: 근데 기초공제 신청서류라고 해서 뭐가 뭔지 잘 모르겠더...
역번역 완료: 기초공제 신청서류라길래 뭔지 잘 이해가 안 갔어....
역번역 시작: A: 그거 작년에 네 아빠가 일했을 때 썼던 거랑 비슷...
역번역 완료: A: 그거 작년에 네 아버지가 일하실 때 사용하셨던 것...
역번역 시작: 필요하면 내가 퇴근하고 다시 한번 같이 확인해줄게....
역번역 완료: 필요하면 내가 퇴근 후에 다시 한 번 함께 확인해줄게....
역번역 시작: B: 응, 고마워....
역번역 완료: B: 응, 고마워....
역번역 시작: 엄마 혹시 오늘 저녁에 뭐 먹을 건데?...
역번역 완료: 엄마, 오늘 저녁에 뭘 먹을 예정이야?...
역번역 시작: 나 늦을 것

In [4]:
import pandas as pd

# 원본 CSV 로딩
df = pd.read_csv("시나리오수정중국어증강.csv")

# 역번역 행 생성
df_bt = df.copy()
df_bt["file_id"] = df_bt["file_id"].astype(str) + "_ch"
df_bt["text"] = df_bt["back_translated"]  # 역번역 텍스트를 text로 이동

# 불필요한 컬럼 삭제 (원한다면)
df_bt = df_bt[["file_id", "is_phishing", "text"]]  # back_translated 제거
df_orig = df[["file_id", "is_phishing", "text"]]   # 원본도 동일한 형태로 정리

df= pd.read_csv("시나리오수정증강합본1.csv")
# 두 데이터프레임 합치기
df_combined = pd.concat([df, df_bt], ignore_index=True)

# 결과 확인 (또는 저장)
df_combined.to_csv("시나리오수정증강합본2.csv", index=False, encoding="utf-8-sig")